In [1]:
# ==========================================
# 1. 自動安裝所需的第三方套件
# ==========================================
print("正在安裝必要套件...")
!pip install Flask pyngrok line-bot-sdk requests google-genai --quiet
print("套件安裝完成！")

# ==========================================
# 2. 匯入模組與讀取 Colab 秘密鑰匙 (Secrets)
# ==========================================
import os
import requests
from flask import Flask, request, abort
from pyngrok import ngrok
from google.colab import userdata
from google import genai

# 讀取環境變數（請確保 Colab 左側「鑰匙」圖示內有設定這些變數）
ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

# ==========================================
# 3. 初始化 Gemini API 用戶端
# ==========================================
# 使用最新版 google-genai SDK 建立用戶端實例
client = genai.Client(api_key=gemini_api_key)

def stateless_query(payload):
    """無狀態查詢函式：將單次問題發送給 Gemini 2.5 Flash 並回傳文字解答"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=payload
    )
    return response.text

# ==========================================
# 4. 清理舊連線並啟動 Ngrok 隧道
# ==========================================
print("正在清理舊的 Ngrok 連線並重新建立隧道...")
ngrok.kill() # 強制關閉舊隧道，避免超過免費版連線數限制

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(port, name="linebot_tunnel")
webhook_url = tunnel.public_url
print(f"Ngrok 隧道建立成功！公網網址為: {webhook_url}")

# ==========================================
# 5. 自動將新網址更新至 LINE 官方後台
# ==========================================
def update_line_webhook(url_to_update):
    """使用 LINE Messaging API 自動更新 Webhook Endpoint"""
    api_url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {"endpoint": url_to_update}

    response = requests.put(api_url, headers=headers, json=data)
    if response.status_code == 200:
        print(f"LINE Webhook URL 已成功自動同步為：{url_to_update}")
        return True
    else:
        print(f"LINE Webhook 更新失敗：{response.status_code} - {response.text}")
        return False

update_line_webhook(webhook_url)

# ==========================================
# 6. 初始化 LINE Bot v3 SDK 與 Flask 伺服器
# ==========================================
from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import Configuration, ApiClient, MessagingApi, ReplyMessageRequest, TextMessage
from linebot.v3.webhooks import MessageEvent, TextMessageContent

app = Flask(__name__)
configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

@app.route("/", methods=['POST'])
def callback():
    # 取得 LINE 傳來的數位簽章，用以驗證請求是否合法
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("收到 LINE Webhook Body: ", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("數位簽章驗證失敗，請檢查 Token 與 Secret 設定。")
        abort(400)
    return 'OK'

# 當收到「文字訊息」時的處理邏輯
@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    print("收到訊息事件 Event: ", event)

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 【0701 核心邏輯判斷】
        # 檢查使用者的訊息是否以 "AI "（大寫 AI 加上一個半形空格）開頭
        if text.startswith('AI '):
            # 擷取 "AI " 之後的所有文字作為 Prompt 提示詞
            prompt = text[3:]
            # 呼叫 Gemini 模型獲取回答
            reply_text = stateless_query(prompt)

            # 回覆 Gemini 生成的內容
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )
        else:
            # 【繼承自 0501 的經典邏輯】
            # 如果不是以 "AI " 開頭，則觸發「鸚鵡學舌連發兩次」機制
            # 在單次回覆中同時塞入兩個 TextMessage，這就是為什麼非 AI 訊息會回應兩次的原因
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text), # 第一條回覆訊息
                        TextMessage(text=event.message.text)  # 第二條回覆訊息
                    ]
                )
            )

# ==========================================
# 7. 啟動 Flask 服務
# ==========================================
if __name__ == "__main__":
    print(f"正在本機 Port {port} 啟動 Flask 伺服器...")
    app.run(port=port)

正在安裝必要套件...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 6.4 MB/s eta 0:00:00
✅ 套件安裝完成！
正在清理舊的 Ngrok 連線並重新建立隧道...


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://strode-clothes-crested.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}
